<a href="https://colab.research.google.com/github/IlyaShlapak2003/-HomeWork7/blob/master/%D0%A2%D1%80%D0%B5%D1%82%D0%B8%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 3 кейс

**В этом кейсе вы будете рассчитывать:**
* retention
* rolling retention
* lifetime
* churn rate
* mau
* wau
* dau

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [1]:
!wget https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv

!wget https://gist.github.com/Vs8th/aacb80595d1d6aaa2e31eb735f8bc644/raw/entries.csv

!wget https://gist.github.com/Vs8th/0e827e9a608117345dd6585ab81e8c86/raw/metrics.txt

--2026-11-01 20:20:47--  https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.github.com (gist.github.com)... 140.82.114.4
Connecting to gist.github.com (gist.github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv [following]
--2026-11-01 20:20:48--  https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14918 (15K) [text/plain]
Saving to: ‘registrations.csv’

registrations.csv   100%[===================>]  14.57K  --.-KB/s    in 0s      

2026-11-01 20:20:48 (73.1 M

In [5]:
import csv
from datetime import datetime, date, timedelta
from collections import defaultdict
def load_csv(filename, date_column):
    result = []
    with open(filename, encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")
        for row in reader:
            parsed_date = date.fromisoformat(
                row[date_column].strip()[:10]
            )
            result.append({
                "user_id": row["user_id"].strip(),
                "date": parsed_date
            })
    return result
registrations = load_csv(
    "registrations.csv", "registration_date"
)
entries = load_csv(
    "entries.csv", "entry_date"
)
registration_dates = {
    row["user_id"]: row["date"]
    for row in registrations
}
user_entries = defaultdict(set)
for row in entries:
    user_entries[row["user_id"]].add(row["date"])
all_users = set(registration_dates)
last_date = max(row["date"] for row in entries)
print("Регистраций:", len(all_users))
print("Записей о входах:", len(entries))
print("Последняя дата:", last_date)

Регистраций: 1000
Записей о входах: 20734
Последняя дата: 2021-12-31


Файлами для работы являются `registrations.csv` и `entries.csv`. В них хранятся данные о регистрациях пользователей и входа на платформу соответственно.

### **Посчитайте Retention 15 дня (в процентах) для пользователей, зарегистрированных в январе**

Cохраните результат в переменную `retention_15_day`

**Примечание:** результат округлите до 5 знаков после запятой

In [6]:
january_users = {
    user_id
    for user_id, reg_date in registration_dates.items()
    if reg_date.year == 2021 and reg_date.month == 1
}
returned = sum(
    registration_dates[user_id] + timedelta(days=15)
    in user_entries[user_id]
    for user_id in january_users
)
retention_15_day = round(
    returned / len(january_users) * 100, 5
) if january_users else 0
print(retention_15_day)

54.65116


In [7]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
# Открываем файл с правильными ответами
with open('metrics.txt', 'r') as f:
    answers = f.read().split('\n')

correct_answer = float(answers[0])

try:
    assert retention_15_day == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Rolling-retention 30 дня (в процентах) для пользователей из той же когорты**

Сохраните результат в переменную `rolling_retention`

**Примечание:** результат округлите до 5 знаков после запятой

In [8]:
returned_rolling = sum(
    any(
        entry_date >= registration_dates[user_id] + timedelta(days=30)
        for entry_date in user_entries[user_id]
    )
    for user_id in january_users
)
rolling_retention = round(
    returned_rolling / len(january_users) * 100, 5
) if january_users else 0
print(rolling_retention)

29.06977


In [9]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[1])

try:
    assert rolling_retention == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Lifetime по всем пользователям, посчитанный как интеграл от n-day retention**

Сохраните результат в переменную `lifetime`

**Примечание:** результат округлите до 5 знаков после запятой

In [16]:
# Ваше решение
total_users = len(registration_dates)
max_days = max(
    (last_date - reg_date).days
    for reg_date in registration_dates.values()
)
retention_values = []
for day in range(max_days + 1):
    returned = sum(
        registration_dates[user_id] + timedelta(days=day)
        in user_entries[user_id]
        for user_id in registration_dates
    )
    retention = returned / total_users
    retention_values.append(retention)
lifetime = round(sum(retention_values), 5)
print(lifetime)

14.804


In [17]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[2])

try:
    assert lifetime == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Churn rate 29 дня (в долях), посчитанный по всем пользователям**

Сохраните результат в переменную `churn_29`.  
Если будете считать CR от Retention, ведите расчет от Rolling Retention, таким образом, мы получим всех, кто не заходил в 29 день и после. Ведя расчет от обычного Retention, наоборот - получим только CR в 29 день.


In [24]:
# Ваше решение
total_users = len(registration_dates)
returned_29 = sum(
    any(
        entry_date >= registration_dates[user_id] + timedelta(days=29)
        for entry_date in user_entries[user_id]
    )
    for user_id in registration_dates
)
churn_29 = round(1 - returned_29 / total_users, 5)
print(churn_29)

0.509


In [25]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[3])

try:
    assert churn_29 == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau за последний месяц/неделю/день записей**

Сохраните результат в переменные `dec_mau`, `dec_wau`, `dec_dau` соответственно

**Примечание:** последний месяц записей - декабрь. Поэтому `mau` рассчитываем для декабря (2021 года), для `wau` берем последнюю неделю - с 25 по 31 декабря, и для `dau` соответственно последний день - 31 декабря.

In [26]:
# Ваше решение
dec_mau = len({
    row["user_id"] for row in entries
    if row["date"].year == 2021
    and row["date"].month == 12
})
print(dec_mau)

133


In [27]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[4])

try:
    assert dec_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [28]:
# Ваше решение
dec_wau = len({
    row["user_id"] for row in entries
    if date(2021, 12, 25) <= row["date"] <= date(2021, 12, 31)
})
print(dec_wau)

84


In [31]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[5])

try:
    assert dec_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [29]:
# Ваше решение
dec_dau = len({
    row["user_id"] for row in entries
    if row["date"] == date(2021, 12, 31)
})
print(dec_dau)

47


In [30]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[6])

try:
    assert dec_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau усредненные**

Сохраните результат в переменные `avg_mau`, `avg_wau`, `avg_dau` соответственно

**Примечание:** результаты округлите до 5 знаков после запятой

In [32]:
# Ваше решение
from collections import defaultdict
monthly_users = defaultdict(set)
for row in entries:
    d = row["date"]
    monthly_users[(d.year, d.month)].add(row["user_id"])
avg_mau = round(sum(len(users) for users in monthly_users.values()) / len(monthly_users), 5)
print(avg_mau)

102.58333


In [33]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[7])

try:
    assert avg_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [34]:
# Ваше решение
weekly_users = defaultdict(set)
for row in entries:
    d = row["date"]
    year, week, _ = d.isocalendar()
    weekly_users[(year, week)].add(row["user_id"])
avg_wau = round(sum(len(users) for users in weekly_users.values()) / len(weekly_users),5)
print(avg_wau)

89.86792


In [35]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[8])

try:
    assert avg_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [36]:
# Ваше решение
daily_users = defaultdict(set)
for row in entries:
    daily_users[row["date"]].add(row["user_id"])
avg_dau = round(sum(len(users) for users in daily_users.values()) / len(daily_users), 5)
print(avg_dau)

40.5589


In [37]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[9])

try:
    assert avg_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
